# IMPORT LIBRARIES


In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import pandas as pd
import re
import string
import os

# DATA PREPARATION


In [3]:
def load_dataset(file_path):
    """
    Membaca file imdb_labelled.txt
    Format tiap baris: <teks>\t<label>
    """
    texts = []
    labels = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line_split = line.strip().split('\t')
            if len(line_split) == 2:
                text, label = line_split
                texts.append(text)
                labels.append(int(label))
    return texts, labels

In [4]:
def text_cleaning(text):
    """
    Membersihkan teks sederhana: lower, hapus angka & tanda baca, dsb.
    """
    text = text.lower()
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)  # hapus tanda baca
    text = re.sub(r'\d+', '', text)  # hapus angka
    text = re.sub(r'\s+', ' ', text)  # hapus spasi berlebih
    text = text.strip()
    return text

In [5]:
def build_vocab(cleaned_texts, min_freq=1):
    """
    Membuat word2idx sederhana berdasarkan frekuensi kemunculan kata.
    """
    freq_dict = {}
    for text in cleaned_texts:
        for word in text.split():
            freq_dict[word] = freq_dict.get(word, 0) + 1

    # sort by frequency
    sorted_words = sorted(freq_dict.items(), key=lambda x: x[1], reverse=True)

    # Buat word2idx, sisipkan token khusus <PAD> dan <UNK>
    word2idx = {'<PAD>': 0, '<UNK>': 1}
    idx = 2
    for word, freq in sorted_words:
        if freq >= min_freq:
            word2idx[word] = idx
            idx += 1
    return word2idx

In [6]:
def encode_text(text, word2idx, max_len=50):
    """
    Mengubah teks menjadi list of token-id (integer),
    dan memotong/padding sampai max_len.
    """
    tokens = text.split()
    encoded = []
    for token in tokens:
        encoded.append(word2idx.get(token, word2idx['<UNK>']))
    # potong jika panjang > max_len
    encoded = encoded[:max_len]
    # padding jika panjang < max_len
    if len(encoded) < max_len:
        encoded += [word2idx['<PAD>']] * (max_len - len(encoded))
    return encoded

In [7]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len=50):
        self.texts = texts
        self.labels = labels
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoded_text = encode_text(text, self.word2idx, self.max_len)
        return torch.LongTensor(encoded_text), torch.LongTensor([label])

# MARKOV & HIDDEN MARKOV MODEL DEFINITION


In [8]:
class MarkovClassifier(nn.Module):
    """
    Markov-like model (sederhana).
    Idenya: Satu "lapisan" transformasi seolah-olah adalah transition probabilities.
    """
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128, pooling='max', num_classes=2):
        super(MarkovClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Kita definisikan "transition" sederhana:
        # Menerima embedding, mengeluarkan representasi (hidden_size)
        self.transition = nn.Linear(embed_dim, hidden_size)

        self.pooling = pooling  # 'max' or 'avg'
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        """
        x: (batch_size, seq_len)
        """
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        # transform token embedding -> "transition states"
        # shape: (batch_size, seq_len, hidden_size)
        states = self.transition(embedded)
        states = torch.tanh(states)  # seolah non-linear activation

        # pooling over seq_len
        if self.pooling == 'max':
            pooled, _ = torch.max(states, dim=1)
        else:  # 'avg'
            pooled = torch.mean(states, dim=1)

        logits = self.fc(pooled)
        return logits

In [9]:
class HiddenMarkovClassifier(nn.Module):
    """
    HMM-like model (lebih 'dalam' dari MarkovClassifier).
    Kita tambahkan semacam 'transition' berulang untuk meniru hidden states.
    """
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128, num_layers=2,
                 pooling='max', num_classes=2):
        super(HiddenMarkovClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Kita definisikan "transition" multi-layer (num_layers).
        # Tiap layer: Linear -> Tanh
        layers = []
        in_size = embed_dim
        for i in range(num_layers):
            layers.append(nn.Linear(in_size, hidden_size))
            in_size = hidden_size
        self.transitions = nn.ModuleList(layers)

        self.pooling = pooling
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        """
        x: (batch_size, seq_len)
        """
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)

        # Lakukan transition berulang (num_layers) untuk setiap token
        # Bentuk: (batch_size, seq_len, hidden_size)
        out = embedded
        for layer in self.transitions:
            out = torch.tanh(layer(out))

        # pooling
        if self.pooling == 'max':
            pooled, _ = torch.max(out, dim=1)
        else:
            pooled = torch.mean(out, dim=1)

        logits = self.fc(pooled)
        return logits

# TRAINING & EVALUATION UTILS


In [10]:
def calculate_accuracy(y_pred, y_true):
    predicted = torch.argmax(y_pred, dim=1)
    correct = (predicted == y_true.view(-1)).sum().item()
    total = y_true.size(0)
    return correct / total

In [11]:
def train_one_epoch(model, dataloader, optimizer, criterion, device='cpu'):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch.view(-1))
        loss.backward()
        optimizer.step()

        acc = calculate_accuracy(outputs, y_batch)
        running_loss += loss.item() * x_batch.size(0)
        running_acc  += acc * x_batch.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc

In [12]:
def validate_one_epoch(model, dataloader, criterion, device='cpu'):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch.view(-1))

            acc = calculate_accuracy(outputs, y_batch)
            running_loss += loss.item() * x_batch.size(0)
            running_acc  += acc * x_batch.size(0)
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc

# EARLY STOPPING


In [13]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.0):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.delta = delta

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# TRAIN AND EVALUATE


In [14]:
def train_and_evaluate(model,
                       train_loader,
                       val_loader,
                       optimizer,
                       scheduler,
                       criterion,
                       epochs=10,
                       patience=5,
                       device='cpu'):

    early_stopper = EarlyStopping(patience=patience)

    best_val_loss = float('inf')
    best_model_state = None

    for epoch in range(1, epochs+1):
        # Training
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        # Validation
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # Update scheduler (misal ReduceLROnPlateau pakai val_loss)
        if scheduler is not None:
            scheduler.step(val_loss)

        print(f"Epoch [{epoch}/{epochs}] | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        # Simpan model terbaik
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()

        # Early Stopping
        early_stopper(val_loss)
        if early_stopper.early_stop:
            print("Early stopping triggered!")
            break

    # Kembalikan model terbaik
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model

# EXPERIMENT CONFIG


In [15]:
from google.colab import drive
import pandas as pd
from google.colab import files
drive.mount('/content/drive')

FILE_PATH = "/content/drive/MyDrive/Machine Learning/imdb_labelled.txt"
BATCH_SIZE = 32
MAX_LEN = 50
EMBED_DIM = 128
DEVICE = 'cpu'  # ubah ke 'cuda' jika ingin pakai GPU (jika tersedia)

HIDDEN_SIZES = [64, 128]
POOLINGS = ['max', 'avg']
OPTIMIZERS = ['SGD', 'RMSProp', 'Adam']
EPOCHS_LIST = [5, 50, 100, 250, 350]

# EarlyStopping
PATIENCE = 5

Mounted at /content/drive


# MAIN EXPERIMENT WITH LOGGING


In [ ]:
def main_experiment_markov():
    # Load & Preprocess
    texts, labels = load_dataset(FILE_PATH)
    cleaned_texts = [text_cleaning(t) for t in texts]
    word2idx = build_vocab(cleaned_texts, min_freq=1)
    vocab_size = len(word2idx)

    # Train-Val Split
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        cleaned_texts, labels, test_size=0.2, random_state=42
    )

    # Dataset & Dataloader
    train_dataset = IMDBDataset(train_texts, train_labels, word2idx, max_len=MAX_LEN)
    val_dataset   = IMDBDataset(val_texts,   val_labels,   word2idx, max_len=MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

    criterion = nn.CrossEntropyLoss()

    # Hasil logging
    results = []

    # Kita akan coba dua jenis model: MarkovClassifier & HiddenMarkovClassifier
    model_types = ['Markov', 'HiddenMarkov']

    for model_type in model_types:
        for hidden_size in HIDDEN_SIZES:
            for pooling in POOLINGS:
                for opt_name in OPTIMIZERS:
                    for epochs in EPOCHS_LIST:
                        print("================================================")
                        print(f"[INFO] Model={model_type}, hidden_size={hidden_size}, "
                              f"pooling={pooling}, optimizer={opt_name}, epochs={epochs}")

                        # Definisikan model
                        if model_type == 'Markov':
                            model = MarkovClassifier(
                                vocab_size=vocab_size,
                                embed_dim=EMBED_DIM,
                                hidden_size=hidden_size,
                                pooling=pooling,
                                num_classes=2
                            ).to(DEVICE)
                        else:  # HiddenMarkov
                            # Anggap num_layers=2 (bisa Anda ganti)
                            model = HiddenMarkovClassifier(
                                vocab_size=vocab_size,
                                embed_dim=EMBED_DIM,
                                hidden_size=hidden_size,
                                num_layers=2,
                                pooling=pooling,
                                num_classes=2
                            ).to(DEVICE)

                        # Definisikan optimizer
                        if opt_name == 'SGD':
                            optimizer = optim.SGD(model.parameters(), lr=0.01)
                        elif opt_name == 'RMSProp':
                            optimizer = optim.RMSprop(model.parameters(), lr=0.001)
                        else:  # 'Adam'
                            optimizer = optim.Adam(model.parameters(), lr=0.001)

                        # Scheduler
                        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                            optimizer, mode='min', factor=0.5, patience=2
                        )

                        # Train & Evaluate
                        trained_model = train_and_evaluate(
                            model,
                            train_loader,
                            val_loader,
                            optimizer,
                            scheduler,
                            criterion,
                            epochs=epochs,
                            patience=PATIENCE,
                            device=DEVICE
                        )

                        # Cek final performance (Train & Val)
                        final_train_loss, final_train_acc = validate_one_epoch(trained_model, train_loader, criterion, DEVICE)
                        final_val_loss,   final_val_acc   = validate_one_epoch(trained_model, val_loader,   criterion, DEVICE)

                        # Simpan ke results
                        results.append({
                            "model_type"      : model_type,
                            "hidden_size"     : hidden_size,
                            "pooling"         : pooling,
                            "optimizer"       : opt_name,
                            "epochs"          : epochs,
                            "final_train_loss": final_train_loss,
                            "final_train_acc" : final_train_acc,
                            "final_val_loss"  : final_val_loss,
                            "final_val_acc"   : final_val_acc
                        })

                        print("================================================\n")

    files.download('experiment_markov_hmm_results.csv')

if __name__ == "__main__":
    main_experiment_markov()

[INFO] Model=Markov, hidden_size=64, pooling=max, optimizer=SGD, epochs=5
Epoch [1/5] | Train Loss: 0.7069, Train Acc: 0.5088 | Val Loss: 0.6958, Val Acc: 0.5200
Epoch [2/5] | Train Loss: 0.6956, Train Acc: 0.5150 | Val Loss: 0.6939, Val Acc: 0.5150
Epoch [3/5] | Train Loss: 0.6911, Train Acc: 0.5262 | Val Loss: 0.6952, Val Acc: 0.4750
Epoch [4/5] | Train Loss: 0.6881, Train Acc: 0.5563 | Val Loss: 0.6932, Val Acc: 0.5250
Epoch [5/5] | Train Loss: 0.6849, Train Acc: 0.5513 | Val Loss: 0.6910, Val Acc: 0.5450

[INFO] Model=Markov, hidden_size=64, pooling=max, optimizer=SGD, epochs=50
Epoch [1/50] | Train Loss: 0.7045, Train Acc: 0.4963 | Val Loss: 0.6892, Val Acc: 0.5400
Epoch [2/50] | Train Loss: 0.6922, Train Acc: 0.5075 | Val Loss: 0.6925, Val Acc: 0.4950
Epoch [3/50] | Train Loss: 0.6886, Train Acc: 0.5262 | Val Loss: 0.6825, Val Acc: 0.5450
Epoch [4/50] | Train Loss: 0.6848, Train Acc: 0.5463 | Val Loss: 0.6813, Val Acc: 0.5600
Epoch [5/50] | Train Loss: 0.6825, Train Acc: 0.5613 |